# 원 작업자용 분할·병합 셀

**이 노트북 자체에서 실행하지 말고 코드 셀을 현재 실행 중인 기존 노트북에 복사하세요.** 기존 6번을 정지하고 저장 종료를 기다린 뒤 첫 셀을 실행합니다. RUN_NAME·ZIP·run_config는 변경하지 않습니다. 기존 7번은 전체 18,764건 기준이므로 병합 전에는 미완료로 나오는 것이 정상입니다.

## 내 담당 앞 15,764건만 재개

In [ ]:
# 기존에 사용하던 노트북에서 실행하세요. 새 빈 세션에서는 먼저 기존 1~5번을 실행해야 합니다.
from itertools import islice
from journal import Journal
from runner import read_inputs, run

OWNER_COUNT = 15764
existing = Journal(run_root / "full", run_config)
print("기존 결과:", existing.summary())
full_journal = run(
    islice(read_inputs(WORK / "inputs.jsonl.gz"), OWNER_COUNT),
    annotator, run_root / "full", run_config,
)
print("내 담당 앞 15,764건 완료. 이후 팀원 결과와 합치세요.")


## 팀원 완료 후 병합

팀원 결과 ZIP과 함께 제공한 `team_split.py`를 업로드하세요. 중복은 원 작업자 결과를 유지하고 보고하며, 누락이 있으면 최종 파일 생성을 중단합니다. 원 체크포인트는 덮어쓰지 않습니다.

In [ ]:
# 기존 노트북에서 내 담당 처리가 끝난 후 실행합니다. 모델을 다시 로드할 필요는 없습니다.
import importlib.util
import json
import zipfile
from pathlib import Path
from google.colab import files
from journal import Journal, atomic_json
from runner import read_inputs

# 이 셀을 사용하려면 team_split.py 파일도 업로드합니다.
print("team_split.py 및 팀원이 보내준 thinkguard_team_last3000_results.zip을 선택하세요.")
incoming = files.upload()
if "team_split.py" not in incoming or "thinkguard_team_last3000_results.zip" not in incoming:
    raise ValueError("두 파일을 모두 선택하세요.")
helper_path = Path("/content/thinkguard_team_merge_helper.py")
helper_path.write_bytes(incoming["team_split.py"])
spec = importlib.util.spec_from_file_location("team_merge_helper", helper_path)
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
zip_path = Path("/content/thinkguard_team_results.zip")
zip_path.write_bytes(incoming["thinkguard_team_last3000_results.zip"])
with zipfile.ZipFile(zip_path) as archive:
    if set(archive.namelist()) != {"annotations.jsonl", "run_config.json", "assignment.json"}:
        raise ValueError("Unexpected result ZIP entries")
    team_config = json.loads(archive.read("run_config.json"))
    assignment = json.loads(archive.read("assignment.json"))
    team_records = [json.loads(line) for line in archive.read("annotations.jsonl").decode("utf-8").splitlines()]
owner_config = json.loads((run_root / "full/run_config.json").read_text(encoding="utf-8"))
saved = Journal(run_root / "full", owner_config)
merged, report = helper.merge_records(read_inputs(WORK / "inputs.jsonl.gz"), saved.records.values(),
                                       team_records, owner_config, team_config, assignment)
merged_dir = run_root / "merged_with_team"
merged_dir.mkdir(exist_ok=True)
atomic_json(merged_dir / "coverage.json", report)
print(json.dumps(report, ensure_ascii=False, indent=2))
if report["missing_ids"]:
    raise RuntimeError("누락이 있어 최종 결과를 생성하지 않았습니다. coverage.json을 확인하세요.")
target = merged_dir / "annotations.jsonl"
temporary = target.with_suffix(".tmp")
with temporary.open("w", encoding="utf-8") as stream:
    for record in merged:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
temporary.replace(target)
atomic_json(merged_dir / "provenance.json", {"owner_config": owner_config, "team_config": team_config,
                                            "assignment": assignment})
print("18,764건 병합 결과:", target)
